# Parte 3: Modelado, Evaluación y Exportación

**Proyecto:** Predicción de Deserción Universitaria (Sistemas Inteligentes - UPN)  
**Dataset:** UCI #697 — Predict Students' Dropout and Academic Success  
**Objetivo de esta entrega (P3):**
1. Entrenar y comparar **4 modelos** con Validación Cruzada Estratificada sobre `X_train`.
2. Seleccionar el **modelo ganador** por F1-Score de la clase Dropout.
3. Evaluar el ganador sobre `X_test` **una única vez** (regla de oro).
4. Calcular la **importancia de variables por permutación**.
5. Exportar el pipeline ganador con **joblib** y definir el umbral de clasificación.

## 0. Configuración e Importaciones

In [ ]:
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import joblib

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score, precision_score,
    recall_score, roc_auc_score, ConfusionMatrixDisplay
)

# 4 Modelos del proyecto
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from src.config import SEED, FIGURES, DATA_PROCESSED
from src.data_prep import get_train_test
from src.preprocessor import get_preprocessor, ALL_FEATURE_COLS

# Semilla global de numpy
np.random.seed(SEED)

print("✔ Librerías importadas correctamente.")
print(f"  SEED = {SEED} | Modelos a comparar: 4")

## 1. Carga de Datos (partición acordada en P2)

In [ ]:
X_train, X_test, y_train, y_test = get_train_test()

print(f"X_train : {X_train.shape} | y_train : {y_train.shape}")
print(f"X_test  : {X_test.shape}  | y_test  : {y_test.shape}")
print(f"Dropout en Train: {y_train.mean()*100:.2f} % ({y_train.sum()} alumnos)")
print(f"Dropout en Test : {y_test.mean()*100:.2f} % ({y_test.sum()} alumnos)")

# Integridad del split acordado
assert len(X_train) == 3539 and len(X_test) == 885
assert not set(X_train.index) & set(X_test.index), 'ERROR: Fuga de índices'
print("\n✔ Split verificado: 3539 train / 885 test, sin solapamiento.")

## 2. Definición de los 4 Modelos y sus Hiperparámetros

Cada modelo se encapsula en un `sklearn.Pipeline` junto al preprocesador de P2,  
garantizando que el ajuste del escalador y el OHE ocurra **solo dentro del fold de train**  
en cada iteración de la CV → cero *data leakage*.

In [ ]:
MODELOS = {
    "Logistic Regression": Pipeline([
        ("preprocessor", get_preprocessor()),
        ("classifier", LogisticRegression(
            C=1.0,
            max_iter=1000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=SEED
        ))
    ]),
    "Decision Tree": Pipeline([
        ("preprocessor", get_preprocessor()),
        ("classifier", DecisionTreeClassifier(
            max_depth=8,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", get_preprocessor()),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            max_depth=None,
            min_samples_leaf=2,
            class_weight="balanced",
            n_jobs=-1,
            random_state=SEED
        ))
    ]),
    "Gradient Boosting": Pipeline([
        ("preprocessor", get_preprocessor()),
        ("classifier", GradientBoostingClassifier(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.8,
            random_state=SEED
        ))
    ]),
}

print("Modelos registrados:")
for nombre, pipe in MODELOS.items():
    clf = pipe.named_steps['classifier']
    print(f"  • {nombre}: {clf.__class__.__name__}")

## 3. Validación Cruzada Estratificada (5-Fold) sobre X_train

**Regla fundamental:** El conjunto de prueba (`X_test`) **no se toca** en esta sección.  
La CV opera exclusivamente sobre `X_train` para seleccionar el mejor modelo.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = ["f1", "precision", "recall", "accuracy", "roc_auc"]

resultados_cv = {}

for nombre, pipeline in MODELOS.items():
    print(f"⏳ Entrenando {nombre}...")
    scores = cross_validate(
        pipeline, X_train, y_train,
        cv=CV,
        scoring=SCORING,
        return_train_score=False,
        n_jobs=-1
    )
    resultados_cv[nombre] = scores
    f1_mean = scores['test_f1'].mean()
    f1_std  = scores['test_f1'].std()
    print(f"   ✔ F1-CV = {f1_mean:.4f} ± {f1_std:.4f}")

print("\n✔ Validación cruzada completada.")

## 4. Tabla Comparativa de Resultados (CV)

In [ ]:
filas = []
for nombre, scores in resultados_cv.items():
    filas.append({
        "Modelo": nombre,
        "F1 (CV)": round(scores['test_f1'].mean(), 4),
        "F1 ± std": f"{scores['test_f1'].mean():.4f} ± {scores['test_f1'].std():.4f}",
        "Precision (CV)": round(scores['test_precision'].mean(), 4),
        "Recall (CV)": round(scores['test_recall'].mean(), 4),
        "Accuracy (CV)": round(scores['test_accuracy'].mean(), 4),
        "ROC-AUC (CV)": round(scores['test_roc_auc'].mean(), 4),
    })

df_cv = pd.DataFrame(filas).sort_values("F1 (CV)", ascending=False).reset_index(drop=True)
df_cv.index += 1  # ranking desde 1

print("=== TABLA COMPARATIVA — Validación Cruzada (5-Fold) sobre X_train ===")
print(f"Métrica de selección: F1-Score (clase Dropout = 1)\n")
display(df_cv)

# Identificar ganador
GANADOR_NOMBRE = df_cv.iloc[0]["Modelo"]
print(f"\n🏆 MODELO GANADOR: {GANADOR_NOMBRE}  (F1-CV = {df_cv.iloc[0]['F1 (CV)']:.4f})")

## 5. Gráfico de Barras — Comparación F1 con Barras de Error

In [ ]:
nombres   = list(resultados_cv.keys())
f1_means  = [resultados_cv[n]['test_f1'].mean() for n in nombres]
f1_stds   = [resultados_cv[n]['test_f1'].std()  for n in nombres]
colores   = ['#e45756' if n == GANADOR_NOMBRE else '#4c78a8' for n in nombres]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(nombres, f1_means, yerr=f1_stds, capsize=5,
              color=colores, edgecolor='white', linewidth=1.2)
ax.axhline(0.78, ls='--', color='orange', lw=1.5, label='Meta F1 > 0.78')
ax.set_ylabel('F1-Score (Dropout) — CV 5-Fold', fontsize=11)
ax.set_title('Comparación de Modelos — Validación Cruzada', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
for bar, mean, std in zip(bars, f1_means, f1_stds):
    ax.text(bar.get_x() + bar.get_width()/2, mean + std + 0.015,
            f'{mean:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()

FIGURES.mkdir(parents=True, exist_ok=True)
plt.savefig(FIGURES / 'comparacion_modelos_cv.png', dpi=200)
plt.show()
print(f"✔ Guardado: {FIGURES / 'comparacion_modelos_cv.png'}")

## 6. Reentrenamiento del Modelo Ganador sobre TODO X_train

Una vez elegido el ganador por CV, se reentrena usando la totalidad de `X_train`  
para aprovechar el máximo de datos antes de la evaluación final.

In [ ]:
pipeline_ganador = MODELOS[GANADOR_NOMBRE]
pipeline_ganador.fit(X_train, y_train)

print(f"✔ Pipeline ganador '{GANADOR_NOMBRE}' reentrenado sobre X_train completo ({len(X_train)} muestras).")

## 7. Evaluación Final sobre X_test (UNA SOLA VEZ)

> ⚠️ **Esta celda se ejecuta una única vez.** El test set es el juez final imparcial.  
> No se modifica ningún hiperparámetro a partir de estos resultados.

In [ ]:
y_pred  = pipeline_ganador.predict(X_test)
y_proba = pipeline_ganador.predict_proba(X_test)[:, 1]

metricas_test = {
    "F1-Score (Dropout)": f1_score(y_test, y_pred),
    "Precision (Dropout)": precision_score(y_test, y_pred),
    "Recall (Dropout)": recall_score(y_test, y_pred),
    "Accuracy": accuracy_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba),
}

print(f"=== EVALUACIÓN FINAL — {GANADOR_NOMBRE} sobre X_test ({len(X_test)} muestras) ===")
for metrica, valor in metricas_test.items():
    estado = '✔' if (metrica == 'F1-Score (Dropout)' and valor >= 0.78) else ' '
    print(f"  {estado} {metrica:<25}: {valor:.4f}")

print("\n--- Reporte Completo por Clase ---")
print(classification_report(y_test, y_pred, target_names=['Resto (0)', 'Dropout (1)']))

## 8. Matriz de Confusión

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Resto (0)', 'Dropout (1)'])
disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
ax.set_title(f'Matriz de Confusión — {GANADOR_NOMBRE}\n(Test: {len(X_test)} muestras)', fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES / 'matriz_confusion_ganador.png', dpi=200)
plt.show()
print(f"✔ Guardado: {FIGURES / 'matriz_confusion_ganador.png'}")

tn, fp, fn, tp = cm.ravel()
print(f"\n  Verdaderos Negativos (TN) = {tn}  |  Falsos Positivos (FP) = {fp}")
print(f"  Falsos Negativos  (FN) = {fn}  |  Verdaderos Positivos (TP) = {tp}")

## 9. Importancia de Variables por Permutación

La importancia por permutación es **modelo-agnóstica**: mide cuánto cae el F1  
cuando los valores de una variable se barajan aleatoriamente (rompiendo su relación con el target).

In [ ]:
# Transformar X_test con el preprocesador del pipeline ganador
preprocessor_fit = pipeline_ganador.named_steps['preprocessor']
clf_fit          = pipeline_ganador.named_steps['classifier']

X_test_trans  = preprocessor_fit.transform(X_test)
feature_names = preprocessor_fit.get_feature_names_out()

print(f"Calculando importancia por permutación ({len(feature_names)} columnas, 30 repeticiones)...")
perm_imp = permutation_importance(
    clf_fit, X_test_trans, y_test,
    n_repeats=30,
    scoring='f1',
    random_state=SEED,
    n_jobs=-1
)

df_imp = pd.DataFrame({
    'Variable': feature_names,
    'Importancia Media': perm_imp.importances_mean,
    'Std': perm_imp.importances_std
}).sort_values('Importancia Media', ascending=False).reset_index(drop=True)

print("\n=== TOP 15 Variables más importantes (Permutation Importance) ===")
display(df_imp.head(15))

## 10. Gráfico de Importancia por Permutación (Top 15)

In [ ]:
top15 = df_imp.head(15)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(
    top15['Variable'][::-1],
    top15['Importancia Media'][::-1],
    xerr=top15['Std'][::-1],
    color='#4c78a8', edgecolor='white', capsize=4
)
ax.set_xlabel('Caída promedio en F1 (Dropout) al permutar la variable', fontsize=10)
ax.set_title(f'Top 15 — Importancia por Permutación\n{GANADOR_NOMBRE}', fontsize=12, fontweight='bold')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.savefig(FIGURES / 'permutation_importance_top15.png', dpi=200)
plt.show()
print(f"✔ Guardado: {FIGURES / 'permutation_importance_top15.png'}")

## 11. Análisis del Umbral de Clasificación

El umbral por defecto en scikit-learn es **0.50**. Dado el contexto (detección temprana  
de deserción), se analiza si un umbral más bajo aumenta el Recall sin sacrificar demasiado la Precisión.

In [ ]:
from sklearn.metrics import precision_recall_curve

UMBRAL_OPERATIVO = 0.40  # umbral institucional del simulador

precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
f1_thresholds = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
umbral_optimo = thresholds[np.argmax(f1_thresholds)]

# Comparación de umbrales
filas_umbral = []
for umbral, etiqueta in [(0.50, 'Default (0.50)'), (UMBRAL_OPERATIVO, f'Operativo ({UMBRAL_OPERATIVO})'), (round(umbral_optimo,2), f'Óptimo F1 ({round(umbral_optimo,2)})')]:
    y_pred_u = (y_proba >= umbral).astype(int)
    filas_umbral.append({
        'Umbral': etiqueta,
        'F1 (Dropout)': round(f1_score(y_test, y_pred_u), 4),
        'Precision': round(precision_score(y_test, y_pred_u, zero_division=0), 4),
        'Recall': round(recall_score(y_test, y_pred_u), 4),
        'Accuracy': round(accuracy_score(y_test, y_pred_u), 4),
    })

df_umbral = pd.DataFrame(filas_umbral)
print(f"Umbral óptimo por F1: {umbral_optimo:.4f}")
display(df_umbral)

# Curva Precision-Recall
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(recalls[:-1], precisions[:-1], color='#4c78a8', lw=2, label='Curva P-R')
ax.axvline(recalls[:-1][np.argmax(f1_thresholds)], ls='--', color='#e45756', lw=1.5, label=f'Umbral óptimo F1 = {umbral_optimo:.2f}')
ax.axvline(recalls[:-1][np.searchsorted(-thresholds, -UMBRAL_OPERATIVO)], ls=':', color='orange', lw=1.5, label=f'Umbral operativo = {UMBRAL_OPERATIVO}')
ax.set_xlabel('Recall (Sensibilidad)', fontsize=11)
ax.set_ylabel('Precision', fontsize=11)
ax.set_title(f'Curva Precision-Recall — {GANADOR_NOMBRE}', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES / 'curva_precision_recall.png', dpi=200)
plt.show()
print(f"✔ Guardado: {FIGURES / 'curva_precision_recall.png'}")

## 12. Exportación del Pipeline Ganador con joblib

In [ ]:
import json

MODELS_DIR = ROOT_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Guardar pipeline
ruta_modelo = MODELS_DIR / 'pipeline_ganador.joblib'
joblib.dump(pipeline_ganador, ruta_modelo)
print(f"✔ Pipeline exportado: {ruta_modelo}")

# Guardar columnas features (para que la app cargue el mismo orden)
ruta_cols = MODELS_DIR / 'feature_cols.json'
with open(ruta_cols, 'w', encoding='utf-8') as f:
    json.dump(ALL_FEATURE_COLS, f, ensure_ascii=False, indent=2)
print(f"✔ Columnas exportadas: {ruta_cols}")

# Guardar métricas de test
ruta_metricas = MODELS_DIR / 'metricas_test.json'
metricas_export = {k: round(float(v), 4) for k, v in metricas_test.items()}
metricas_export['modelo'] = GANADOR_NOMBRE
metricas_export['umbral_operativo'] = UMBRAL_OPERATIVO
metricas_export['umbral_optimo_f1'] = round(float(umbral_optimo), 4)
with open(ruta_metricas, 'w', encoding='utf-8') as f:
    json.dump(metricas_export, f, ensure_ascii=False, indent=2)
print(f"✔ Métricas exportadas: {ruta_metricas}")

# Verificar que se puede recargar correctamente
pipeline_cargado = joblib.load(ruta_modelo)
y_pred_verify = pipeline_cargado.predict(X_test)
assert (y_pred_verify == y_pred).all(), 'ERROR: El pipeline recargado produce resultados distintos'
print("\n✔ Verificación de integridad del joblib: OK (predicciones idénticas).")
print(f"\n{'='*55}")
print(f"  MODELO GANADOR : {GANADOR_NOMBRE}")
print(f"  F1 (Dropout)   : {metricas_test['F1-Score (Dropout)']:.4f}")
print(f"  ROC-AUC        : {metricas_test['ROC-AUC']:.4f}")
print(f"  Umbral operativo: {UMBRAL_OPERATIVO}")
print(f"{'='*55}")

## 13. Resumen Final — Tabla Comparativa completa (CV + Test ganador)

Esta tabla es la que se reporta en la **Sección 4.3** del informe.

In [ ]:
print("=== RESUMEN FINAL P3 ===")
print()
print("--- Comparación por CV (5-Fold, X_train) ---")
display(df_cv[['Modelo', 'F1 ± std', 'Precision (CV)', 'Recall (CV)', 'Accuracy (CV)', 'ROC-AUC (CV)']])

print(f"\n--- Evaluación Final del Ganador (X_test, 1 sola vez) ---")
print(f"  Modelo: {GANADOR_NOMBRE}")
for k, v in metricas_test.items():
    print(f"  {k:<25}: {v:.4f}")

print(f"\n--- Archivos generados ---")
print(f"  models/pipeline_ganador.joblib")
print(f"  models/feature_cols.json")
print(f"  models/metricas_test.json")
print(f"  reports/figures/comparacion_modelos_cv.png")
print(f"  reports/figures/matriz_confusion_ganador.png")
print(f"  reports/figures/permutation_importance_top15.png")
print(f"  reports/figures/curva_precision_recall.png")